In [1]:
import pandas as pd

In [2]:
# Load dataset
df_asli = pd.read_csv('aug_datafix.csv')
df_aug = pd.read_csv('smote_datafix.csv')

In [4]:
# Merge final dengan kolom konsisten + kolom tambahan
base_cols = [
    'IDJwb', 'IDPSJ', 'questions', 'answerKeys', 'answer',
    'raw_grade', 'max_grade', 'grade', 'labela', 'label'
 ]
out_cols = base_cols + ['is_synthetic', 'inverse_words']

# Ambil hanya kolom inti dari data asli
df_asli_core = df_asli[base_cols].copy()

# Normalisasi jawaban asli agar satu baris (untuk inverse_words data asli)
asli_lookup = (
    df_asli_core.assign(
        answer_single_line=df_asli_core['answer']
        .astype(str)
        .str.replace(r'[\r\n]+', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )
    .drop_duplicates(subset=['IDJwb'])
    .set_index('IDJwb')
 )

rows = []
for r in df_aug[['IDJwb', 'IDPSJ', 'grade', 'is_synthetic', 'inverse_words']].itertuples(index=False):
    row = {c: '' for c in out_cols}

    # Kolom minimal yang selalu ada di data smote
    row['IDJwb'] = r.IDJwb
    row['IDPSJ'] = r.IDPSJ
    row['grade'] = r.grade
    row['is_synthetic'] = r.is_synthetic
    row['inverse_words'] = r.inverse_words if pd.notna(r.inverse_words) else ''

    # Untuk data asli, isi kolom full dari df_asli
    if str(r.is_synthetic) == '0' and r.IDJwb in asli_lookup.index:
        src = asli_lookup.loc[r.IDJwb]
        for c in [x for x in base_cols if x != 'IDJwb']:
            row[c] = src[c] if pd.notna(src[c]) else ''
        row['inverse_words'] = src['answer_single_line'] if pd.notna(src['answer_single_line']) else ''

    rows.append(row)

df_merge = pd.DataFrame(rows, columns=out_cols)
df_merge.to_csv('merge_datafix.csv', index=False)

print('Total baris:', len(df_merge))
print('Total kolom:', len(df_merge.columns))
print('Jumlah data sintesis:', (df_merge['is_synthetic'].astype(str) == '1').sum())

Total baris: 1743
Total kolom: 12
Jumlah data sintesis: 691
